# Lab 21-06 · Oscillators and VCOs

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import vco

db10 = lambda x: 10 * math.log10(x)
K_B, T = 1.380649e-23, 300.0
kT = K_B * T
VT = kT / 1.602176634e-19


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Amplitude and frequency

*Predict* the amplitude from the hard-limit formula and from the describing function, and the frequency from $1/(2\pi\sqrt{LC})$.

In [2]:
f0, L = 5e9, 1e-9
t = vco.tank(f0, L, 15, 50)
C, Q, Rp = t["C"], t["Q"], t["Rp"]
I0, V0 = 1e-3, 2 * VT
Vh, Vd = vco.amplitude_hard(Rp, I0), vco.amplitude_df(Rp, I0, V0)
print(f"C = {C*1e12:.4f} pF, Q = {Q:.2f}, Rp = {Rp:.1f} ohm")
print(f"hard limit {Vh:.4f} V, describing function {Vd:.4f} V")
s = ngspice("vco.cir")
print(f"ngspice: amplitude {s['amp']:.4f} V, free-running {s['fa']/1e9:.5f} GHz, injected {s['fb']/1e9:.5f} GHz")
shift = vco.groszkowski_shift(Rp, C, 2 * math.pi * f0, I0, V0, Vd)
print(f"Groszkowski: {shift*100:.4f} % -> {f0*(1+shift)/1e9:.5f} GHz (square-wave limit {-100/(8*Q*Q):.4f} %)")
Vs = np.linspace(0.01, 0.8, 300)
G = [vco.describing_function(x, I0, V0) for x in Vs]
plt.figure(); plt.plot(Vs, np.array(G) * 1e3, label="G(V)"); plt.axhline(1e3 / Rp, ls="--", c="k", label="1/R_p")
plt.xlabel("amplitude V (V)"); plt.ylabel("large-signal transconductance (mS)"); plt.legend(); plt.show()

C = 1.0132 pF, Q = 11.54, Rp = 362.5 ohm
hard limit 0.4615 V, describing function 0.4591 V


ngspice: amplitude 0.4592 V, free-running 4.99668 GHz, injected 5.01000 GHz
Groszkowski: -0.0643 % -> 4.99678 GHz (square-wave limit -0.0939 %)


## Part 2 · The impulse sensitivity function

*Predict* the phase shift per unit charge at 0°, 90°, 180° and 270°.

In [3]:
th = np.linspace(0, 2 * np.pi, 16, endpoint=False)
g = vco.isf(Q, 1.0, Q / 7.01, th)                 # Example 21-06.1 in normalized units (loop gain 7)
print(f"Gamma_rms = {math.sqrt(np.mean(g**2)):.4f} (ideal {1/math.sqrt(2):.4f}), Gamma_dc = {np.mean(g):+.4f}")
g50 = vco.isf(50.0, 1.0, 50.0 / 7.01, [math.pi / 2, 0.0])
print(f"Q = 50: Gamma(90 deg) = {g50[0]:.4f}, Gamma(0) = {g50[1]:+.4f}")
x = np.linspace(0, 2 * np.pi, 400)
plt.figure(); plt.plot(np.degrees(x), -np.sin(x), label="-sin(theta)"); plt.plot(np.degrees(th), g, "o", label="simulated")
plt.xlabel("injection phase (degrees)"); plt.ylabel("Gamma"); plt.legend(); plt.show()

Gamma_rms = 0.7029 (ideal 0.7071), Gamma_dc = -0.0006


Q = 50: Gamma(90 deg) = -1.0003, Gamma(0) = +0.0103


## Part 3 · A phase-noise budget

*Predict* the total at 1 MHz from the tank and devices ($F$ = 2) and from a 1 kΩ resistor on a 400 MHz/V tuning line.

In [4]:
Vd = vco.amplitude_df(Rp, I0, V0)
qmax = C * Vd
df = np.logspace(3, 7, 201)                     # includes 100 kHz and 1 MHz exactly
tank_white = np.array([2 * vco.isf_phase_noise(f, 4 * kT / Rp, qmax) for f in df])
tank_dev = tank_white * (1 + 22.5e3 / df)                 # with the 22.5 kHz 1/f^3 corner
line = np.array([vco.control_noise_pn(400e6, 4 * kT * 1e3, f) for f in df])
line50 = np.array([vco.control_noise_pn(50e6, 4 * kT * 1e3, f) for f in df])
at = lambda arr, f: db10(np.interp(f, df, arr))
print(f"1 MHz, white parts: tank+devices {at(tank_white,1e6):.2f}, line {at(line,1e6):.2f}, "
      f"total {at(tank_white+line,1e6):.2f} dBc/Hz (flicker adds {at(tank_dev+line,1e6)-at(tank_white+line,1e6):.2f} dB)")
print(f"100 kHz, white parts: total {at(tank_white+line,1e5):.2f} dBc/Hz; with flicker {at(tank_dev+line,1e5):.2f}")
plt.figure()
for arr, lab_ in ((tank_dev, "tank + devices (F = 2, 22.5 kHz corner)"), (line, "tuning line, 400 MHz/V"),
                  (line50, "tuning line, 50 MHz/V"), (tank_dev + line, "total, 400 MHz/V")):
    plt.semilogx(df, 10 * np.log10(arr), label=lab_)
plt.xlabel("offset (Hz)"); plt.ylabel("L (dBc/Hz)"); plt.legend(fontsize=8); plt.show()

1 MHz, white parts: tank+devices -115.73, line -118.78, total -113.98 dBc/Hz (flicker adds 0.06 dB)
100 kHz, white parts: total -93.98 dBc/Hz; with flicker -93.37


## Part 4 · Band map

*Predict* how $K_{VCO}$ changes from code 0 to code 7.

In [5]:
plt.figure()
for code_ in range(8):
    cmin = 0.70e-12 + code_ * 60e-15 + 0.15e-12
    vc = np.linspace(0, 1, 50)
    f = [vco.freq(L, cmin + 0.15e-12 * (1 - x)) for x in vc]      # varactor falls linearly with V_ctrl
    plt.plot(vc, np.array(f) / 1e9, label=f"code {code_}")
    if code_ in (0, 7):
        print(f"code {code_}: {f[0]/1e9:.3f} to {f[-1]/1e9:.3f} GHz, average K_VCO {(f[-1]-f[0])/1e6:.0f} MHz/V")
plt.xlabel("control voltage (V)"); plt.ylabel("frequency (GHz)"); plt.legend(fontsize=7, ncol=2); plt.show()

code 0: 5.033 to 5.459 GHz, average K_VCO 426 MHz/V
code 7: 4.224 to 4.466 GHz, average K_VCO 242 MHz/V


## Part 5 · RC-CR quadrature

*Predict* the worst IRR over 2.40–2.50 GHz with RC at ±10%.

In [6]:
fs = np.linspace(2.40e9, 2.50e9, 101)
plt.figure()
for rc in (0.9, 1.0, 1.1):
    C0 = rc / (2 * math.pi * 2.45e9 * 1e3)
    irr = []
    for f in fs:
        i_, q_ = vco.rc_cr(f, 1e3, C0)
        eps = abs(q_ / i_) - 1
        irr.append(min(80.0, vco.irr_db(eps, 0)) if abs(eps) > 1e-12 else 80.0)   # plot capped at 80 dB
    print(f"RC x {rc}: worst {min(irr):.2f} dB")
    plt.plot(fs / 1e9, irr, label=f"RC x {rc}")
plt.xlabel("frequency (GHz)"); plt.ylabel("IRR (dB, capped at 80)"); plt.legend(); plt.show()
s = ngspice("polyphase.cir")
print(f"ngspice at 2.40 GHz: ratio {s['rdb']:.4f} dB, phase difference {s['dph']:.4f} deg")

RC x 0.9: worst 24.03 dB
RC x 1.0: worst 39.74 dB
RC x 1.1: worst 24.78 dB
ngspice at 2.40 GHz: ratio -0.1791 dB, phase difference 90.0000 deg


## Part 6 · Injection locking and Adler's equation

*Predict* whether injection at 0.9 and at 2 lock ranges locks, and the beat frequency if not.

In [7]:
Qa, Va, r = 10.0, 10.0 / 3, 0.05
w_free = vco.free_running_shift(Qa, 1.0, Va)
amp = math.hypot(*vco.settle(Qa, 1.0, Va))
wL = vco.lock_range(1 / (2 * math.pi), Qa, r) * 2 * math.pi
print(f"free-running offset {w_free:.2e}, lock range {wL:.4e}, predicted beat at 2 wL {math.sqrt(3)*wL:.4e}")
plt.figure()
for m in (0.9, 2.0):
    rate, ts, psi = vco.injection_beat(Qa, 1.0, Va, r * amp / Qa, w_free + m * wL, cycles=1500, skip=300)
    print(f"injection at {m} lock ranges: slip rate {rate:+.4e}")
    plt.plot(ts / (2 * math.pi), psi - psi[0], label=f"{m} x lock range")
plt.xlabel("cycles"); plt.ylabel("relative phase psi (rad)"); plt.legend(); plt.show()

free-running offset -4.71e-04, lock range 2.5031e-03, predicted beat at 2 wL 4.3355e-03


injection at 0.9 lock ranges: slip rate -9.2497e-06


injection at 2.0 lock ranges: slip rate -4.2579e-03


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.